# 01 - Data Loading & Schema Profiling

**Goal**: Load all raw CSV tables, profile every column (dtype, null rate, unique count, samples),
and document the complete data dictionary.

**Learning concepts**: Pandas data loading, dtype specification, memory management for large CSVs,
understanding relational data schemas.

---

In [ ]:
import pandas as pd
import numpy as np
from talentlens.config import (
    POSTINGS_CSV, COMPANIES_CSV, COMPANY_INDUSTRIES_CSV,
    COMPANY_SPECIALITIES_CSV, EMPLOYEE_COUNTS_CSV,
    BENEFITS_CSV, JOB_INDUSTRIES_CSV, JOB_SKILLS_CSV,
    SALARIES_CSV, INDUSTRIES_MAP_CSV, SKILLS_MAP_CSV,
)
from talentlens.dataset import load_raw_postings, load_secondary_tables

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Load the Main Postings Table

The main dataset has **3.38M rows** and is ~493MB. We load it with specified dtypes
to avoid pandas guessing wrong types (which wastes memory).

> **Tip**: For development, use `nrows=10000` to load a small sample first.
> Once your code works, remove the limit to process the full dataset.

In [ ]:
# Load a small sample first to understand the schema
# Change nrows=None to load the full dataset when ready
df_postings = load_raw_postings(nrows=10000)

# .info() is your best friend for a first look at any DataFrame:
# - Shows every column name, dtype, and non-null count in one shot
# - Shows total memory usage at the bottom
# - Immediately tells you what's missing and what types pandas assigned
df_postings.info()

In [ ]:
## 2. Profile the Postings Columns — Going Deeper

`.info()` gave us the basics (dtypes, non-null counts). Now we build a **profile table**
that adds two things `.info()` doesn't show:
- **null_pct**: What percentage is missing? (more intuitive than raw counts)
- **most_common**: What's the most frequent value? (reveals data skew — if 90% of a column is one value, it's not very useful as a feature)

def profile_dataframe(df: pd.DataFrame, name: str = "DataFrame") -> pd.DataFrame:
    """Generate a profiling summary for every column in a DataFrame."""
    # Find the most common value per column (reveals skew)
    most_common = []
    for col in df.columns:
        top_val = df[col].mode()
        if len(top_val) > 0:
            most_common.append(top_val.iloc[0])
        else:
            most_common.append(None)

    profile = pd.DataFrame({
        'dtype': df.dtypes,
        'non_null': df.count(),
        'null_count': df.isna().sum(),
        'null_pct': (df.isna().sum() / len(df) * 100).round(1),
        'unique': df.nunique(),
        'most_common': most_common,
    })
    print(f"\n{'='*60}")
    print(f"  {name}: {len(df):,} rows x {len(df.columns)} columns")
    print(f"{'='*60}")
    return profile

In [ ]:
# Profile goes deeper than .info() — look at null_pct and most_common
profile_postings = profile_dataframe(df_postings, "Postings")
profile_postings

In [ ]:
## 3. Load & Profile All Secondary Tables

The dataset is **relational** — like a database with multiple linked tables.
If you've taken a Database Management class, this is an ER (Entity-Relationship) schema:

```
postings (PK: job_id)
    ├──1:N── job_skills    (FK: job_id, FK: skill_abr)  → skills (PK: skill_abr)
    ├──1:N── job_industries (FK: job_id, FK: industry_id) → industries (PK: industry_id)
    ├──1:N── benefits      (FK: job_id)
    ├──1:N── salaries      (FK: job_id)
    └──N:1── companies     (PK: company_id)
                ├──1:N── company_industries
                ├──1:N── company_specialities
                └──1:N── employee_counts
```

**Why this matters for our project**: These secondary tables let us **enrich** each posting
with skills, benefits, company size, and industry data. These become **features** for our
ML models in Phase 3 (e.g., "does having more benefits correlate with higher salary?").

## 3. Load & Profile All Secondary Tables

The dataset is **relational** — like a database with multiple linked tables:

```
postings (main) ──┬── job_skills ──── skills (mapping)
                   ├── job_industries ── industries (mapping)
                   ├── benefits
                   ├── salaries
                   └── companies ──┬── company_industries
                                   ├── company_specialities
                                   └── employee_counts
```

Tables are linked via `job_id` or `company_id` foreign keys.

In [ ]:
tables = load_secondary_tables()

for name, df in tables.items():
    display(profile_dataframe(df, name))

## 4. Explore the Mapping Tables

These are small lookup tables that decode IDs into human-readable names.

In [ ]:
# Skills mapping: 35 skill categories
print("=== Skills Mapping ===")
display(tables['skills_map'])

print(f"\n=== Industries Mapping (first 20 of {len(tables['industries_map'])}) ===")
display(tables['industries_map'].head(20))

## 5. Join job_skills with Skills Names

The `job_skills` table uses abbreviations like `IT`, `SALE`, `DSGN`.
Let's join with the mapping table to see the full skill names.

In [ ]:
# Join skills abbreviations to full names
skills_enriched = tables['job_skills'].merge(
    tables['skills_map'],
    on='skill_abr',
    how='left'
)
print(f"Job-skill associations: {len(skills_enriched):,}")
print(f"\nTop 15 skills by frequency:")
skills_enriched['skill_name'].value_counts().head(15)

# Summary: focus on what matters — null rates for our research themes
import os
csv_size_mb = os.path.getsize(POSTINGS_CSV) / 1e6

print("=== Dataset Summary ===")
print(f"Raw CSV file size: {csv_size_mb:.0f} MB")
print(f"Sample loaded: {len(df_postings):,} rows")

print(f"\nSecondary tables:")
for name, tbl in tables.items():
    print(f"  {name}: {len(tbl):,} rows x {tbl.shape[1]} cols")

# These are the columns that matter for our 4 research themes:
# - description → NLP/RAG (all themes)
# - med_salary → Theme 1 (salary prediction)
# - applies, views → Theme 2 (ghost jobs) & Theme 4 (engagement)
# - formatted_experience_level → Theme 3 (entry-level paradox)
print(f"\nKey null rates in postings (sample):")
key_cols = ['description', 'title', 'med_salary', 'remote_allowed', 
            'formatted_experience_level', 'skills_desc', 'applies', 'views']
for col in key_cols:
    if col in df_postings.columns:
        null_pct = df_postings[col].isna().mean() * 100
        print(f"  {col}: {null_pct:.1f}% null")

In [ ]:
# Get total line count using Python (cross-platform, no wc needed)
import os
csv_size_mb = os.path.getsize(POSTINGS_CSV) / 1e6
# We already know from the sample how many columns there are;
# for total rows, use the loaded sample or note the known count.
total_lines = "~3,383,602 (from CSV header count)"

print("=== Dataset Summary ===")
print(f"Raw CSV file size: {csv_size_mb:.0f} MB")
print(f"Total postings (raw CSV lines): {total_lines}")
print(f"Sample loaded: {len(df_postings):,} rows")
print(f"Columns: {df_postings.shape[1]}")
print(f"\nSecondary tables:")
for name, df in tables.items():
    print(f"  {name}: {len(df):,} rows x {df.shape[1]} cols")

print(f"\nKey null rates in postings (sample):")
key_cols = ['description', 'title', 'med_salary', 'remote_allowed', 
            'formatted_experience_level', 'skills_desc', 'applies', 'views']
for col in key_cols:
    if col in df_postings.columns:
        null_pct = df_postings[col].isna().mean() * 100
        print(f"  {col}: {null_pct:.1f}% null")

## Key Observations

*(Fill in after running the notebook)*

1. **Postings table**: X rows, Y columns. Description null rate = Z%.
2. **Salary coverage**: Only X% of postings have salary data.
3. **Skills coverage**: X job-skill associations across Y unique skills.
4. **Next step**: Clean the data in notebook 02.

---

**→ Next notebook**: `02-mp-data-cleaning.ipynb`